# Chapter 6 · Advanced RAG

Improve the retrieval of a RAG system and measure how much each piece adds: BM25, hybrid search with reciprocal rank fusion, reranking with a *cross-encoder*, and query rewriting and HyDE with the language model. The knowledge base has 82 fragments (FAQ, terms and catalog) and the test, 40 queries of four types.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. The notebook loads three models: the language model, the embedding model and the reranking model (about 2.3 GB). Without the variants that use the language model, it takes about a minute.

The full explanation is in the PDF chapter. This notebook is the lab.

📘 The book on Amazon: [paperback](https://www.amazon.com/dp/B0HMX9N1KQ) · [Kindle](https://www.amazon.com/dp/B0HMX44VJ4)

## 1. Install BM25, load the models and the data

In [ ]:
!pip install -q rank_bm25

In [ ]:
"""Chapter 6: Advanced RAG. Hybrid search, reranking and query rewriting."""
import json
import re
import unicodedata
import urllib.request
from pathlib import Path

import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-0.6B"             # without a GPU: smaller and faster
MODEL = None                              # None: choose according to the hardware
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
RERANKING_MODEL = "BAAI/bge-reranker-v2-m3"
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/en/"



def load(name=None, with_language=True):
    """The language, embedding and reranking models."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if name is None:
        name = GPU_MODEL if device == "cuda" else CPU_MODEL
    tok = model = None
    if with_language:
        tok = AutoTokenizer.from_pretrained(name)
        model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
        model.to(device).eval()
    embeddings = SentenceTransformer(EMBEDDING_MODEL, device=device)
    reranker = CrossEncoder(RERANKING_MODEL, device=device)
    print(f"Models on {device}")
    return tok, model, embeddings, reranker


def download(name):
    """A file from the data folder; in Colab, it is downloaded from the repository."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return path.read_text(encoding="utf-8")


def knowledge_base():
    """FAQ, terms and conditions and catalog, with an identifier for each
    fragment."""
    fragments = []
    for q in json.loads(download("faq.json"))["questions"]:
        fragments.append({"id": f"faq-{q['id']}",
                          "text": q["question"] + " " + q["answer"]})
    terms = download("terms_and_conditions.md")
    for section in re.split(r"\n(?=## )", terms)[1:]:
        title, text = section.strip().split("\n", 1)
        number = title.split(".")[0][3:]
        fragments.append({"id": f"cond-{number}",
                          "text": title[3:] + ". " + text.strip()})
    for p in json.loads(download("catalog.json")):
        fragments.append({"id": p["ref"], "text": (
            f"{p['name']} (reference {p['ref']}), {p['category']}, "
            f"${p['price']}. {p['description']}")})
    return fragments

In [ ]:
tok, model, embeddings, reranker = load(MODEL)
fragments = knowledge_base()
texts = {f["id"]: f["text"] for f in fragments}
queries = json.loads(download("retrieval_queries.json"))
print(len(fragments), "fragments and", len(queries), "queries")
print(texts["UR-5310"])

## 2. Two searchers

In [ ]:
class SemanticSearcher:
    """Embeddings, as in chapter 4. Returns the identifiers in order."""

    def __init__(self, fragments):
        self.ids = [f["id"] for f in fragments]
        self.matrix = embeddings.encode(["passage: " + f["text"] for f in fragments],
                                        normalize_embeddings=True)

    def rank(self, query, prefix="query: "):
        vector = embeddings.encode([prefix + query], normalize_embeddings=True)
        return [self.ids[i] for i in np.argsort(-(self.matrix @ vector[0]))]


STOPWORDS = set(
    "a an the and or of to in on at for with from by is are be do does i me my you "
    "your we us it its they this that not no so if as can have has any some something "
    "anything how what which when where who much many there s d m t ll re ve".split())


def words(text):
    """Lowercase, without accents and without stopwords. References such as
    UR-5310 are kept whole."""
    text = "".join(c for c in unicodedata.normalize("NFD", text.lower())
                   if unicodedata.category(c) != "Mn")
    return [w for w in re.findall(r"[a-z]{2}-\d{4}|[a-z0-9]+", text)
            if w not in STOPWORDS]


class BM25Searcher:
    """Keyword search with BM25."""

    def __init__(self, fragments):
        self.ids = [f["id"] for f in fragments]
        self.bm25 = BM25Okapi([words(f["text"]) for f in fragments])

    def rank(self, query, with_zeros=False):
        """Fragments that share no word with the query score 0, and their
        order means nothing: they are dropped."""
        scores = self.bm25.get_scores(words(query))
        order = np.argsort(-scores, kind="stable")
        return [self.ids[i] for i in order if with_zeros or scores[i] > 0]

In [ ]:
semantic, bm25 = SemanticSearcher(fragments), BM25Searcher(fragments)
query = "What's the price of reference UR-5310?"
print("Words:", words(query))
print("Embeddings:", semantic.rank(query)[:3])
print("BM25:", bm25.rank(query)[:3])

## 3. Fuse and rerank

In [ ]:
def fuse(*rankings, k=60):
    """Reciprocal rank fusion (RRF): each list adds 1 / (k + position)."""
    scores = {}
    for ranking in rankings:
        for position, ident in enumerate(ranking, 1):
            scores[ident] = scores.get(ident, 0) + 1 / (k + position)
    return sorted(scores, key=scores.get, reverse=True)


def rerank(query, candidates, texts, n=20):
    """A reranking model scores each (query, fragment) pair of the first n
    candidates; the rest stay behind, in their order."""
    first = candidates[:n]
    scores = reranker.predict([(query, texts[i]) for i in first])
    return [first[i] for i in np.argsort(-scores)] + candidates[n:]

In [ ]:
query = "If I'm not home when the delivery driver comes, what happens?"
hybrid = fuse(semantic.rank(query), bm25.rank(query))
print("Hybrid:", hybrid[:3])
print("Reranked:", rerank(query, hybrid, texts)[:3])

## 4. Rewrite the query

In [ ]:
def ask(system, user, examples=(), max_tokens=120):
    """Sends a conversation to the model and returns its answer as text."""
    messages = [{"role": "system", "content": system}]
    for question, answer in examples:
        messages += [{"role": "user", "content": question},
                     {"role": "assistant", "content": answer}]
    messages.append({"role": "user", "content": user})
    text = tok.apply_chat_template(messages, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    answer = tok.decode(new, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()


REWRITE = ("You are the search engine of North Cycles, a bicycle store. "
           "Rewrite the customer's query as a clear, complete search sentence, "
           "using the words the store would use. Keep references and product "
           "names. Do not answer the query: write only the search sentence.")


HYPOTHETICAL = ("You write the website of North Cycles, a bicycle store. In two "
                "sentences, write the answer to the customer's query the way the "
                "store's website or terms of sale would word it. If you don't "
                "know the exact details, make them up.")


REWRITE_EXAMPLES = [
    ("my new tire got a flat",
     "Does the warranty cover a flat on a new tire?"),
    ("that shipping to alaska and hawaii thing",
     "How much does shipping to Alaska and Hawaii cost and how long does it take?"),
    ("Does the CA-4100 have a visor?",
     "Does the helmet with reference CA-4100 have a visor?")]


HYPOTHETICAL_EXAMPLES = [
    ("my new tire got a flat",
     "The warranty doesn't cover the normal wear of parts such as tires and "
     "inner tubes, nor flats. It does cover manufacturing defects for "
     "2 years."),
    ("Does the CA-4100 have a visor?",
     "Trail Helmet (reference CA-4100), helmets, $89. Mountain bike helmet with "
     "an adjustable visor and protection at the back of the head.")]


def rewrite(query):
    """A clearer, more complete query, written by the model."""
    return ask(REWRITE, query, REWRITE_EXAMPLES)


def hypothetical(query):
    """HyDE: a made-up answer, which looks more like the documents than the
    query itself."""
    return ask(HYPOTHETICAL, query, HYPOTHETICAL_EXAMPLES)

In [ ]:
for query in ["that pay-in-installments thing",
              "box came totally smashed up"]:
    print(query)
    print("   Rewritten:", rewrite(query))
    print("   HyDE:", hypothetical(query))

## 5. Measure

In [ ]:
def measure(rankings, queries, ks=(1, 3, 5)):
    """Hits in the top k (recall@k) and mean reciprocal rank (MRR)."""
    result = {f"@{k}": 0 for k in ks}
    rr = 0
    for ranking, q in zip(rankings, queries):
        positions = [i for i, ident in enumerate(ranking, 1)
                     if ident in q["correct"]]
        first = min(positions) if positions else None
        for k in ks:
            result[f"@{k}"] += first is not None and first <= k
        rr += 1 / first if first else 0
    result["MRR"] = round(rr / len(queries), 3)
    return result


def by_type(rankings, queries, k=3):
    """Hits in the top k, split by query type."""
    types = {}
    for ranking, q in zip(rankings, queries):
        hit = any(i in q["correct"] for i in ranking[:k])
        total = types.setdefault(q["type"], [0, 0])
        total[0] += hit
        total[1] += 1
    return {t: f"{h}/{n}" for t, (h, n) in types.items()}

Every variant. To skip the language model, call `run(with_rewriting=False)`.

In [ ]:
def run(with_rewriting=True):
    """Every retrieval variant, measured on the test queries."""
    fragments = knowledge_base()
    texts = {f["id"]: f["text"] for f in fragments}
    queries = json.loads(download("retrieval_queries.json"))
    semantic, bm25 = SemanticSearcher(fragments), BM25Searcher(fragments)
    print(len(fragments), "fragments,", len(queries), "queries")
    sem = [semantic.rank(q["query"]) for q in queries]
    kw = [bm25.rank(q["query"]) for q in queries]
    zeros = [bm25.rank(q["query"], with_zeros=True) for q in queries]
    hyb = [fuse(s, w) for s, w in zip(sem, kw)]
    rer = [rerank(q["query"], h, texts) for q, h in zip(queries, hyb)]
    methods = {"Embeddings": sem, "BM25": kw,
               "Hybrid with BM25 zeros": [fuse(s, w)
                                          for s, w in zip(sem, zeros)],
               "Hybrid": hyb, "Hybrid + reranking": rer}
    if with_rewriting:
        rewritten = [rewrite(q["query"]) for q in queries]
        methods["Rewriting + hybrid + rerank."] = [
            rerank(r, fuse(semantic.rank(r), bm25.rank(r)), texts)
            for r in rewritten]
        hyde = [hypothetical(q["query"]) for q in queries]
        hyde_only = [semantic.rank(h, "passage: ") for h in hyde]
        methods["HyDE only"] = hyde_only
        methods["Hybrid + HyDE + rerank."] = [
            rerank(q["query"], fuse(s, w, h), texts)
            for q, s, w, h in zip(queries, sem, kw, hyde_only)]
    results = {}
    for name, rankings in methods.items():
        results[name] = {**measure(rankings, queries),
                         "by_type": by_type(rankings, queries),
                         "top": [r[:5] for r in rankings]}
        print(f"{name:32s}", {k: v for k, v in results[name].items()
                               if k != "top"})
    if with_rewriting:
        results["rewritten_queries"] = rewritten
        results["hypothetical_answers"] = hyde
    return results

In [ ]:
results = run()

The failures of the best method, with the top three fragments it returns:

In [ ]:
best = results["Hybrid + reranking"]["top"]
for q, top in zip(queries, best):
    if not any(i in q["correct"] for i in top[:3]):
        print(q["query"], "→", top[:3], "| correct:", q["correct"])

## Exercises

1. Explain in your own words why embeddings fail with product references and why BM25 fails with colloquial queries.

2. Work out by hand the RRF score, with k = 60, of a fragment that comes third in the embeddings and fifth in BM25, and of another that comes first in the embeddings and does not appear in BM25. Which one ends up ahead?

3. Try the fusion with k = 1 and with k = 1000. What happens to the results? Why?

4. Change the number of candidates that are reranked, `n`, to 5, 10 and 40. How do the hits change? And the time?

5. Rerank the embedding results directly, without hybrid search, and check that you get the same as hybrid search with reranking. Repeat the comparison reranking only 5 candidates. When does BM25 help?

6. Improve the descriptions of the locks and of the enduro bike so that they say what they are for, in a customer's words, and measure again. Are they found now?

7. Add each catalog fragment's category and price as metadata. Write a function that, for queries such as "e-bike under $2000", first filters by category and price and then searches among the remaining fragments. How would you extract the conditions from the query? (Hint: chapter 3.)

8. Write five queries that depend on an earlier conversation, such as "what about in size 54?" after asking about the Aero 3. Does any method find them? Rewrite them by hand and compare.

9. Connect the best retrieval to the chapter 5 assistant and repeat its evaluation with the twenty questions. Do the answers improve? Is the fragment for the question about the shipping cost of a partial return retrieved now?

10. **Retrieval report.** Write a short report, about ten lines long, recommending a retrieval setup for North Cycles, backed by the figures from your run and taking into account the time of each variant.

## References

Models [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base) and [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3). BM25: [Robertson and Zaragoza (2009)](https://doi.org/10.1561/1500000019). RRF: Cormack, Clarke and Büttcher (2009). HyDE: [Gao et al. (2022)](https://arxiv.org/abs/2212.10496). The North Cycles store, its catalog and its documents are fictitious. Original code and experiments.